# Noise-aware differentiation in 1D, 2D, and 3D

The caller supplies `noise_std=sigma_prior`, an **a priori estimate** of the
measurement-noise standard deviation, in the same units as the data. The
operator receives observed samples and chooses regularization accordingly.
It does not add noise, estimate its standard deviation from the data, or need a
clean reference. `noise_std=0` preserves the original clean-data operator.

This notebook generates a known noisy dataset solely to measure accuracy.
In use, replace `samples` with your acquired data and set `sigma_prior` from
instrument specifications, separate calibration, or domain knowledge. The
estimate need not be exact. The analytic reference below is only for scoring.

Regularization is selected on the original field independently for each axis,
then held fixed for differentiation. In 2D/3D, transverse axes are smoothed too.
This is a separable tensor fit, not a globally isotropic multidimensional
optimization. The fit uses no exact derivatives or endpoint data.

Install `python -m pip install -e '.[host,notebook,test]'` from the repository root.
Use the corrected BLAS launcher in `docs/corrected_blas_build.md`, or start
with `OMP_NUM_THREADS=1` when using the original development BLAS library.


In [ ]:
import pybspf.operators as bspf_operators
import pybspf.plans as bspf_plans

import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
import pybspf as bspf

sigma = 1e-3  # Used only to construct a reproducible validation dataset.
sigma_prior = 1e-3  # A priori estimate supplied to the operator, in data units.
constructors = (bspf_plans.plan_1d, bspf_plans.plan_2d, bspf_plans.plan_3d)
results = []
for dimension, constructor in enumerate(constructors, start=1):
    coordinates = tuple(jnp.linspace(0, 1, n) for n in (33, 35, 37)[:dimension])
    mesh = jnp.meshgrid(*coordinates, indexing="ij")
    phase = sum((i+1)*x for i, x in enumerate(mesh))
    exact = jnp.stack([(i+1)*jnp.cos(phase) for i in range(dimension)])
    noise = np.random.default_rng(45).normal(size=phase.shape)
    samples = jnp.sin(phase) + sigma*jnp.asarray(noise)
    clean_plan = constructor(*coordinates, degree=5, n_basis=12)
    noisy_plan = constructor(*coordinates, degree=5, n_basis=12, noise_std=sigma_prior)
    raw = jax.jit(bspf_operators.gradient)(clean_plan, samples)
    regularized = jax.jit(bspf_operators.gradient)(noisy_plan, samples)
    errors = [float(jnp.linalg.norm(v-exact)/jnp.linalg.norm(exact))
              for v in (raw, regularized)]
    diagnostics = jax.jit(bspf_operators.noise_diagnostics)(noisy_plan, samples)
    print(f"{dimension}D relative L2: original={errors[0]:.3e}, regularized={errors[1]:.3e}")
    print("Supplied noise prior:", [float(d.noise_std) for d in diagnostics])
    print("alphas:", [float(d.alpha) for d in diagnostics],
          "residual/noise:", [float(d.residual_ratio) for d in diagnostics],
          "at search edge:", [bool(d.at_search_edge) for d in diagnostics])
    assert np.isfinite(errors).all() and errors[1] < errors[0]
    results.append((coordinates[0], exact, raw, regularized))


## Derivative error along a central x-line

The plots show x-derivative errors with other coordinates fixed at their center
indices. Boundary bias can remain even when total error improves. A discrepancy
ratio near one means the selected fit residual matches the assumed noise level;
A selection at a search-grid edge requires reviewing the supplied prior or
`noise_alphas` range. Higher derivatives can need a higher `noise_penalty_order`.

`differentiate(..., order=0)` returns fitted samples in noise-aware mode.
`mixed_partial`, `hessian` and `laplacian` hold selections fixed on the original
field. Repeatedly differentiating an output instead would perform a new
selection with an inappropriate original noise level. Selection is discrete:
JAX autodiff differentiates the selected linear operator, not its argmin.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), constrained_layout=True)
for dimension, (ax, result) in enumerate(zip(axes, results), start=1):
    x, exact, raw, regularized = result
    line = (0, slice(None)) + tuple(n//2 for n in exact.shape[2:])
    ax.plot(x, np.asarray(raw-exact)[line], label="Original BSPF")
    ax.plot(x, np.asarray(regularized-exact)[line], label="Noise-aware BSPF")
    ax.set(title=f"{dimension}D", xlabel="x", ylabel="x-derivative error")
    ax.legend(); ax.grid(alpha=.2)
plt.show()
